<a href="https://colab.research.google.com/github/karthikeyan15-bit/NPL_project_repo/blob/main/NLP_4th_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jackksoncsie/spam-email-dataset")

print("Path to dataset files:", path)

100%|██████████| 2.86M/2.86M [00:00<00:00, 4.47MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jackksoncsie/spam-email-dataset/versions/1


In [2]:
import os

# List the contents of the downloaded dataset directory
dataset_files = os.listdir(path)
print(f"Files in dataset directory: {dataset_files}")

# Assuming there's a CSV file, let's try to find it and construct its full path
csv_files = [f for f in dataset_files if f.endswith('.csv')]

if csv_files:
    data_file_path = os.path.join(path, csv_files[0])
    print(f"Found CSV file: {data_file_path}")
else:
    print("No CSV files found in the dataset directory. Please inspect the directory manually if the data is in a different format.")
    data_file_path = None

Files in dataset directory: ['emails.csv']
Found CSV file: /root/.cache/kagglehub/datasets/jackksoncsie/spam-email-dataset/versions/1/emails.csv


In [3]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
try:
    df = pd.read_csv(data_file_path)
    print("Dataset loaded successfully!")
    print("\nFirst 5 rows of the dataset:")
    print(df.head())
    print("\nDataset Info:")
    df.info()
except Exception as e:
    print(f"Error loading dataset: {e}")

Dataset loaded successfully!

First 5 rows of the dataset:
                                                text  spam
0  Subject: naturally irresistible your corporate...     1
1  Subject: the stock trading gunslinger  fanny i...     1
2  Subject: unbelievable new homes made easy  im ...     1
3  Subject: 4 color printing special  request add...     1
4  Subject: do not have money , get software cds ...     1

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5728 non-null   object
 1   spam    5728 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 89.6+ KB


In [4]:
import re
import string

def clean_text(text):
    text = text.lower()  # Convert text to lowercase
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'<.*?>', '', text) # Remove HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text) # Remove punctuation
    text = re.sub(r'\n', '', text) # Remove newline characters
    text = re.sub(r'\w*\d\w*', '', text) # Remove words containing numbers
    text = re.sub(r'subject', '', text) # Remove the word 'subject'
    return text.strip()

print("Applying text cleaning...")
df['cleaned_text'] = df['text'].apply(clean_text)

print("\nText cleaning complete. Here are the first 5 rows with the new 'cleaned_text' column:")
print(df[['text', 'cleaned_text', 'spam']].head())

Applying text cleaning...

Text cleaning complete. Here are the first 5 rows with the new 'cleaned_text' column:
                                                text  \
0  Subject: naturally irresistible your corporate...   
1  Subject: the stock trading gunslinger  fanny i...   
2  Subject: unbelievable new homes made easy  im ...   
3  Subject: 4 color printing special  request add...   
4  Subject: do not have money , get software cds ...   

                                        cleaned_text  spam  
0  naturally irresistible your corporate identity...     1  
1  the stock trading gunslinger  fanny is merrill...     1  
2  unbelievable new homes made easy  im wanting t...     1  
3  color printing special  request additional inf...     1  
4  do not have money  get software cds from here ...     1  


In [7]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download NLTK data if not already downloaded
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

stop_words = set(stopwords.words('english'))

def tokenize_and_remove_stopwords(text):
    tokens = word_tokenize(text) # Tokenize the text
    filtered_tokens = [word for word in tokens if word not in stop_words] # Remove stop words
    return ' '.join(filtered_tokens)

print("Applying tokenization and stop word removal...")
df['processed_text'] = df['cleaned_text'].apply(tokenize_and_remove_stopwords)

print("\nTokenization and stop word removal complete. Here are the first 5 rows with the new 'processed_text' column:")
print(df[['text', 'cleaned_text', 'processed_text', 'spam']].head())

Applying tokenization and stop word removal...

Tokenization and stop word removal complete. Here are the first 5 rows with the new 'processed_text' column:
                                                text  \
0  Subject: naturally irresistible your corporate...   
1  Subject: the stock trading gunslinger  fanny i...   
2  Subject: unbelievable new homes made easy  im ...   
3  Subject: 4 color printing special  request add...   
4  Subject: do not have money , get software cds ...   

                                        cleaned_text  \
0  naturally irresistible your corporate identity...   
1  the stock trading gunslinger  fanny is merrill...   
2  unbelievable new homes made easy  im wanting t...   
3  color printing special  request additional inf...   
4  do not have money  get software cds from here ...   

                                      processed_text  spam  
0  naturally irresistible corporate identity lt r...     1  
1  stock trading gunslinger fanny merrill muzo 

### Task 2: Preprocess email text (continued) - Feature Extraction

Now, we'll convert the `processed_text` into numerical features using `TfidfVectorizer`.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to 5000 to keep the feature space manageable

# Fit and transform the processed text
X = tfidf_vectorizer.fit_transform(df['processed_text'])

# Get the target variable
y = df['spam']

print("TF-IDF feature extraction complete.")
print(f"Shape of feature matrix (X): {X.shape}")
print(f"Shape of target variable (y): {y.shape}")

TF-IDF feature extraction complete.
Shape of feature matrix (X): (5728, 5000)
Shape of target variable (y): (5728,)


### Task 3: Build spam classification model

Now that our features (`X`) and target (`y`) are ready, we will split the data into training and testing sets and then train a spam classification model. We'll use a `Multinomial Naive Bayes` classifier, which is commonly used and performs well for text classification tasks.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

# Initialize and train the Multinomial Naive Bayes model
model = MultinomialNB()
model.fit(X_train, y_train)

print("Multinomial Naive Bayes model trained successfully.")

Training set size: 4582 samples
Test set size: 1146 samples
Multinomial Naive Bayes model trained successfully.


### Task 4: Evaluate classification accuracy

Now we will evaluate the performance of our trained Multinomial Naive Bayes model using the test set. We'll calculate accuracy, precision, recall, and F1-score to get a comprehensive understanding of the model's effectiveness.

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Evaluation:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Evaluation:
Accuracy: 0.9782
Precision: 0.9853
Recall: 0.9276
F1-Score: 0.9556

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       856
           1       0.99      0.93      0.96       290

    accuracy                           0.98      1146
   macro avg       0.98      0.96      0.97      1146
weighted avg       0.98      0.98      0.98      1146



### Task 5: Generate spam analysis report

Now we will generate a comprehensive report summarizing the NLP-based spam detection system. This report will cover the key steps, the model's performance, and insights from the evaluation.

In [12]:
print("--- Spam Detection System Analysis Report ---")
print("\n1. Project Goal: ")
print("   To build an NLP-based spam detection system for email.")

print("\n2. Data Collection and Preprocessing:")
print(f"   - Dataset: {data_file_path}")
print(f"   - Total emails processed: {len(df)}")
print("   - Preprocessing steps included: Text cleaning (lowercase, URL removal, HTML tag removal, punctuation removal, newline removal, words with numbers removal, 'subject' keyword removal), tokenization, and stop word removal.")
print("   - Feature Extraction: TF-IDF Vectorization (max_features=5000).")

print("\n3. Model Training:")
print("   - Model Used: Multinomial Naive Bayes Classifier.")
print(f"   - Training data size: {X_train.shape[0]} samples")
print(f"   - Test data size: {X_test.shape[0]} samples")

print("\n4. Model Evaluation:")
print(f"   - Accuracy: {accuracy:.4f}")
print(f"   - Precision (Spam): {precision:.4f}")
print(f"   - Recall (Spam): {recall:.4f}")
print(f"   - F1-Score (Spam): {f1:.4f}")

print("\n5. Classification Report:")
print(classification_report(y_test, y_pred))

print("\n6. Conclusion:")
print("   The NLP-based spam detection system demonstrates strong performance with high accuracy, precision, and F1-score, indicating its effectiveness in identifying spam emails. The model shows a very good ability to correctly identify non-spam (recall for class 0 is 1.00) and a high precision for identifying spam (0.9853). There is a slight trade-off in recall for spam (0.9276), meaning a small percentage of actual spam emails might be missed, but overall the model is highly reliable.")

--- Spam Detection System Analysis Report ---

1. Project Goal: 
   To build an NLP-based spam detection system for email.

2. Data Collection and Preprocessing:
   - Dataset: /root/.cache/kagglehub/datasets/jackksoncsie/spam-email-dataset/versions/1/emails.csv
   - Total emails processed: 5728
   - Preprocessing steps included: Text cleaning (lowercase, URL removal, HTML tag removal, punctuation removal, newline removal, words with numbers removal, 'subject' keyword removal), tokenization, and stop word removal.
   - Feature Extraction: TF-IDF Vectorization (max_features=5000).

3. Model Training:
   - Model Used: Multinomial Naive Bayes Classifier.
   - Training data size: 4582 samples
   - Test data size: 1146 samples

4. Model Evaluation:
   - Accuracy: 0.9782
   - Precision (Spam): 0.9853
   - Recall (Spam): 0.9276
   - F1-Score (Spam): 0.9556

5. Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       856
   